In [1]:
# importar pandas e numpy
import pandas as pd
import numpy as np


In [2]:
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

df_roubos = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='iso-8859-1')
#print(df_roubos.head())
#print(df_roubos.describe())
# print(df_roubos['regiao'].unique())

In [7]:

# Delimitar as variaveis cisp região munic roubo_veiculos
df_roubos = df_roubos[['cisp', 'regiao', 'munic', 'roubo_veiculo']].copy()

#corrigindo a grande Niteroi
correcao_acentuacao = df_roubos['regiao'].str.startswith('Grande Niter', na=False)
# print(correcao_acentuacao)
df_roubos.loc[correcao_acentuacao, 'regiao'] = 'Grande Niterói'
print(df_roubos['regiao'].unique())

<StringArray>
['Capital', 'Baixada Fluminense', 'Interior', 'Grande Niterói']
Length: 4, dtype: str


In [11]:
# Agrupando e totalizando os roubos de veiculo por cisp regiao e munic
df_roubos = df_roubos.groupby(['cisp', 'regiao', 'munic'], as_index=False)[['roubo_veiculo']].sum()
display(df_roubos)
# print(df_roubos.head(20))


,cisp,regiao,munic,roubo_veiculo
0,1,Capital,Rio de Janeiro,583
1,4,Capital,Rio de Janeiro,2230
2,5,Capital,Rio de Janeiro,1384
3,6,Capital,Rio de Janeiro,4708
4,7,Capital,Rio de Janeiro,2035
...,...,...,...,...
142,165,Interior,Mangaratiba,504
143,166,Interior,Angra dos Reis,971
144,167,Capital,Paraty,2
145,167,Interior,Paraty,106


In [13]:
# calculando média, total, maior, menor mediana e quartis
array_roubos = np.array(df_roubos['roubo_veiculo'])

media = np.mean(array_roubos)
mediana = np.median(array_roubos)

maximo = np.max(array_roubos)
minimo = np.min(array_roubos)
total = np.sum(array_roubos)

q1 = np.percentile(array_roubos, 25)
q3 = np.percentile(array_roubos, 75)

print(f'Média: {media:.2f}')
print(f'Mediana: {mediana:.2f}')

print(f'Máximo: {maximo:.2f}')
print(f'Mínimo: {minimo:.2f}')
print(f'Total: {total:.2f}')

print(f'Q1: {q1:.2f}')
print(f'Q3: {q3:.2f}')


Média: 4969.84
Mediana: 942.00
Máximo: 38084.00
Mínimo: 2.00
Total: 730567.00
Q1: 90.00
Q3: 7390.00


In [20]:
# Criando uma série com os maiores
df_maiores = df_roubos[df_roubos['roubo_veiculo'] >= q3].copy()
df_maiores['flag'] = 'mais'
display(df_maiores.head())

# Criando uma série com os menores
df_menores = df_roubos[df_roubos['roubo_veiculo'] <= q1].copy()
df_menores['flag'] = 'menos'
display(df_menores.head())



,cisp,regiao,munic,roubo_veiculo,flag
13,17,Capital,Rio de Janeiro,7771,mais
14,18,Capital,Rio de Janeiro,7393,mais
16,20,Capital,Rio de Janeiro,9201,mais
17,21,Capital,Rio de Janeiro,19884,mais
18,22,Capital,Rio de Janeiro,15236,mais


,cisp,regiao,munic,roubo_veiculo,flag
7,11,Capital,Rio de Janeiro,18,menos
41,45,Capital,Rio de Janeiro,27,menos
79,91,Interior,Valença,49,menos
80,92,Interior,Rio das Flores,8,menos
83,94,Interior,Piraí;Pinheiral,53,menos


In [16]:
# Concatenando os dois dataframe df_maiores e df_menores
df_roubos_flag = pd.concat([df_maiores, df_menores], ignore_index=True)

In [19]:
# Exportando os dados para um arquivo CSV utf-8-sig para ler no power bi
df_roubos_flag.to_csv('roubos_flag.csv', index=False, sep=';', encoding='utf-8-sig')